# Triage Agent GRPO — Smoke Test (Colab / Kaggle)

An **OpenEnv** RL environment where an LLM learns to resolve enterprise IT support tickets by calling KB-search and incident-management tools, then submitting a grounded resolution with citations.

- Environment (HF Space): [yahid/triage_agent_env](https://huggingface.co/spaces/yahid/triage_agent_env)
- Trained model (200-step A100 run): [yahid/triage-agent-qwen3b](https://huggingface.co/yahid/triage-agent-qwen3b)

> **This notebook runs 5 GRPO training steps** on `Qwen2.5-0.5B-Instruct` to verify the full environment + reward pipeline on a free T4 GPU (~10-15 min).
> The production 200-step run used `Qwen2.5-3B-Instruct` on an A100.

**Required:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "No GPU detected. Enable GPU: Runtime → Change runtime type → T4 GPU"
)
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
# Qwen2.5-0.5B + GRPO overhead ≈ 6-8 GB — T4 (15 GB) handles this comfortably

In [ ]:
%%capture
!pip install trl transformers accelerate peft rouge-score matplotlib datasets

In [ ]:
import os

REPO_DIR = "/content/triage_env"
if not os.path.exists(REPO_DIR):
    !git clone https://huggingface.co/spaces/yahid/triage_agent_env {REPO_DIR}

# Absolute path — safe to re-run
os.chdir(REPO_DIR)
print(f"cwd: {os.getcwd()}")
!ls

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, ".")

# server/__init__.py auto-imports TriageAgentEnvironment which requires
# openenv-core — a private package not on PyPI. The training code only needs
# server.corpus and server.rewards. Stub __init__.py so those imports work.
Path("server/__init__.py").write_text("# stub: openenv-core not needed for training\n")

from server.corpus import Corpus
from server.rewards import primary_reward, reward_grounding

corpus_check = Corpus(Path("data"))
import json
n_articles = len(corpus_check._articles) if hasattr(corpus_check, "_articles") else "?"
with open("data/train_tickets.json") as _f:
    _n_tickets = len(json.load(_f))

print(f"✓ server.corpus  — {n_articles} KB articles")
print(f"✓ server.rewards — primary_reward, reward_grounding imported")
print(f"✓ data           — {_n_tickets} training tickets")

In [ ]:
# ── vllm_ascend / optional-dep stubs ─────────────────────────────────────────
# TRL's import_utils.py calls find_spec() for several optional packages at
# module level. Stub the ones that might be partially installed to avoid
# ValueError from modules with __spec__=None.
import sys, types, importlib, importlib.util

class _Stub:
    def __init__(self, name="stub"):
        self.__name__ = self.__package__ = name
        self.__version__ = "0.0.0"
        self.__path__ = []
        self.__file__ = "stub"
        self.__spec__ = importlib.util.spec_from_loader(name, loader=None)
    def __getattr__(self, n): return self
    def __call__(self, *a, **k): return self

for _m in ["deepspeed", "unsloth", "liger_kernel", "comet_ml", "mlflow",
           "mergekit", "llm_blender"]:
    if _m not in sys.modules:
        _s = _Stub(_m)
        sys.modules[_m] = _s
        for _sub in ["config", "merge", "chunked_loss", "trainer"]:
            sys.modules[f"{_m}.{_sub}"] = _s

# ── Standard imports ──────────────────────────────────────────────────────────
import json, math, os, random, re
from collections import Counter
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from rouge_score import rouge_scorer

try:
    from transformers import TrainerCallback as _TrainerCallback
except ImportError:
    _TrainerCallback = object

print("✓ All imports OK")

In [ ]:
# ── Smoke-test constants (T4 16 GB safe) ─────────────────────────────────────
MODEL      = "Qwen/Qwen2.5-0.5B-Instruct"   # 0.5B fits T4; prod used 3B on A100
MAX_STEPS  = 5

DATA_DIR   = Path("data")
OUTPUT_DIR = Path("triage_grpo_smoke")
PLOTS_DIR  = Path("assets/plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# VRAM fragmentation guard
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"Model      : {MODEL}")
print(f"Max steps  : {MAX_STEPS}")
print(f"Output dir : {OUTPUT_DIR}")

In [ ]:
# ── Corpus + ticket index (loaded once, shared by all reward functions) ────────
from server.corpus import Corpus

_CORPUS: Optional[Corpus] = None
_TICKET_INDEX: Dict[str, dict] = {}

def get_corpus() -> Corpus:
    global _CORPUS
    if _CORPUS is None:
        _CORPUS = Corpus(DATA_DIR)
    return _CORPUS

def get_ticket_index() -> Dict[str, dict]:
    global _TICKET_INDEX
    if not _TICKET_INDEX:
        with open(DATA_DIR / "train_tickets.json") as f:
            tickets = json.load(f)
        _TICKET_INDEX = {t.get("ticket_id", t.get("id", "")): t for t in tickets}
    return _TICKET_INDEX

# Pre-warm
corpus       = get_corpus()
ticket_index = get_ticket_index()
train_tickets = list(ticket_index.values())
print(f"✓ Corpus ready   — {len(corpus._articles) if hasattr(corpus, '_articles') else '?'} KB articles")
print(f"✓ Ticket index   — {len(train_tickets)} training tickets")

In [ ]:
# ── Tool-call parser ──────────────────────────────────────────────────────────
# Handles all 4 formats Qwen emits:
#   (A) Standard:       {"tool_name": "...", "arguments": {...}}
#   (B) OpenAI:         {"function": {"name": "...", "arguments": "..."}}
#   (C) Dict shorthand: {"submit_resolution": {...args...}}
#   (D) Python call:    submit_resolution({...})
# All four can be wrapped in ```json fences, <tool_call> tags, or bare.

KNOWN_TOOLS = {
    "search_kb", "search_tickets", "search_incidents",
    "get_article", "get_ticket", "get_incident",
    "submit_resolution",
}

_ID_PATTERN = re.compile(r'\b(?:KB|INC|TKT|TRAIN|TICKET)-\d+\b')


def _extract_balanced_json(s: str, start: int) -> Optional[str]:
    if start >= len(s) or s[start] != '{':
        return None
    depth, in_str, escape = 0, False, False
    for i in range(start, len(s)):
        c = s[i]
        if escape:   escape = False; continue
        if c == '\\': escape = True; continue
        if c == '"': in_str = not in_str; continue
        if in_str:   continue
        if c == '{':   depth += 1
        elif c == '}': depth -= 1
        if depth == 0: return s[start:i + 1]
    return None


def _normalize(name: str, args: Any) -> Optional[Dict]:
    if name not in KNOWN_TOOLS: return None
    if isinstance(args, str):
        try:    args = json.loads(args)
        except: args = {}
    return {"tool_name": name, "arguments": args if isinstance(args, dict) else {}}


def _normalize_object(obj: Dict) -> Optional[Dict]:
    name = obj.get("tool_name") or obj.get("name")
    if isinstance(name, str) and name in KNOWN_TOOLS:
        return _normalize(name, obj.get("arguments") or obj.get("parameters") or {})
    fn = obj.get("function")
    if isinstance(fn, dict):
        n = fn.get("name")
        if isinstance(n, str) and n in KNOWN_TOOLS:
            return _normalize(n, fn.get("arguments", {}))
    for tool in KNOWN_TOOLS:
        if tool in obj and isinstance(obj[tool], dict):
            return _normalize(tool, obj[tool])
    return None


def _try_parse_body(body: str) -> Optional[Dict]:
    m = re.search(r'\b(' + '|'.join(KNOWN_TOOLS) + r')\s*\(\s*\{', body)
    if m:
        jp = _extract_balanced_json(body, m.end() - 1)
        if jp:
            try:
                r = _normalize(m.group(1), json.loads(jp))
                if r: return r
            except json.JSONDecodeError: pass
    idx = 0
    while idx < len(body):
        brace = body.find('{', idx)
        if brace == -1: break
        jp = _extract_balanced_json(body, brace)
        if jp:
            try:
                obj = json.loads(jp)
                if isinstance(obj, dict):
                    r = _normalize_object(obj)
                    if r: return r
            except json.JSONDecodeError: pass
            idx = brace + 1
        else: break
    return None


def parse_tool_call(text: str) -> Optional[Dict]:
    if not isinstance(text, str): return None
    for m in re.finditer(r"```(?:json)?\s*\n?(.*?)```", text, re.DOTALL):
        r = _try_parse_body(m.group(1))
        if r: return r
    for m in re.finditer(r"<tool_call>\s*(.*?)\s*</tool_call>", text, re.DOTALL):
        r = _try_parse_body(m.group(1))
        if r: return r
    return _try_parse_body(text)


def _completion_text(completion) -> str:
    if isinstance(completion, str): return completion
    if isinstance(completion, list):
        for msg in reversed(completion):
            if isinstance(msg, dict) and msg.get("role") == "assistant":
                return msg.get("content", "")
    return str(completion)


def _prompt_user_text(prompt) -> str:
    if isinstance(prompt, str): return prompt
    if isinstance(prompt, list):
        for msg in prompt:
            if isinstance(msg, dict) and msg.get("role") == "user":
                return msg.get("content", "")
    return ""


print("✓ Parser ready")

In [ ]:
# ── System prompt + prompt builder ────────────────────────────────────────────

SYSTEM_PROMPT_GROUNDED = """You are an enterprise IT triage agent. You resolve support tickets using ONLY the retrieved context provided in the user message.

You MUST output your answer in this EXACT format — no other format is accepted:

```json
{"tool_name": "submit_resolution", "arguments": {"resolution": "plain text answer here", "cited_artifacts": ["KB-00001"], "confidence": 0.85, "escalate": false}}
```

Critical format rules — violations are penalised:
- The outer wrapper MUST be a ```json ... ``` code fence. Do NOT use <tool_call> tags, Python syntax, or any other format.
- \"resolution\" MUST be a plain string (not a JSON object, not a list, not nested). Write the resolution as prose.
- \"cited_artifacts\" MUST be a JSON array of string IDs (e.g. [\"KB-00001\"]). Cite ONLY IDs that appear in the Retrieved Context section.
- \"confidence\" MUST be a float between 0.0 and 1.0.
- \"escalate\" MUST be a boolean. Set to true only when the context is insufficient to resolve the ticket; in that case cite nothing.
- Output exactly ONE tool call. Do not search or fetch — all relevant context is already provided."""

_ONESHOT = """Example of correct output format:
```json
{"tool_name": "submit_resolution", "arguments": {"resolution": "Verify TCP/179 reachability, check BGP timers, correct any AS or MD5 mismatches.", "cited_artifacts": ["KB-00001"], "confidence": 0.85, "escalate": false}}
```

Now resolve THIS ticket:
"""


def build_training_prompt(ticket: dict, corpus: Corpus, distractor_k: int = 3) -> List[Dict]:
    # Gold articles
    gold_articles = []
    for aid in ticket.get("gold_cited_ids", []):
        art = corpus.get_article(aid) or corpus.get_ticket(aid) or corpus.get_incident(aid)
        if art:
            gold_articles.append(art)

    # Distractor articles from search (realistic noise)
    distractor_hits = corpus.search_kb(ticket["title"], max_results=distractor_k + 2)
    distractors = [
        corpus.get_article(h["article_id"])
        for h in distractor_hits
        if h["article_id"] not in ticket.get("gold_cited_ids", [])
    ]
    distractors = [d for d in distractors if d is not None][:distractor_k]

    # Shuffle so gold position doesn't leak
    context_items = gold_articles + distractors
    random.shuffle(context_items)

    context_block = "\n\n".join(
        f"### {a.get('article_id', a.get('id', ''))}\n{a.get('title', '')}\n"
        f"{a.get('body', a.get('content', ''))[:1000]}"
        for a in context_items
    )

    tid = ticket.get("ticket_id", ticket.get("id", ""))
    return [
        {"role": "system", "content": SYSTEM_PROMPT_GROUNDED},
        {"role": "user", "content": (
            f"{_ONESHOT} # Ticket: {tid}\n"
            f"**Title:** {ticket['title']}\n"
            f"**Description:** {ticket['description']}\n\n"
            f"# Retrieved Context:\n{context_block}\n\n"
            "Resolve this ticket using ONLY the retrieved context. Output exactly one "
            "`submit_resolution` tool call as a JSON code block. "
            "`cited_artifacts` MUST list at least one ID from the Retrieved Context above "
            "(e.g. KB-XXXXX); an empty list [] is only valid when escalate=true."
        )},
    ]


def build_dataset(tickets: List[dict], corpus: Corpus):
    from datasets import Dataset
    rows = []
    for t in tickets:
        tid = t.get("ticket_id", t.get("id", ""))
        rows.append({"prompt": build_training_prompt(t, corpus), "ticket_id": tid})
    return Dataset.from_list(rows)


print("✓ Prompt builder ready")
print("✓ Dataset builder ready")

In [ ]:
# ── Reward helper extractors ───────────────────────────────────────────────────
# Allow partial credit during cold-start when model uses wrong field names.
# Canonical keys still win via a 0.6× discount on fallback paths.

_RESOLUTION_FALLBACK_KEYS = (
    "details", "description", "solution", "resolution_steps",
    "resolution_notes", "notes", "answer", "text",
    "action", "steps_to_resolve", "summary",
)


def _extract_resolution_text(args: dict) -> Tuple[str, bool]:
    """Returns (text, used_canonical_key)."""
    if not isinstance(args, dict): return "", False
    canonical = args.get("resolution")
    if isinstance(canonical, str) and canonical.strip():
        return canonical, True
    if isinstance(canonical, dict):
        nested = canonical.get("resolution") or canonical.get("text") or ""
        if isinstance(nested, str) and nested.strip():
            return nested, False
    for key in _RESOLUTION_FALLBACK_KEYS:
        v = args.get(key)
        if isinstance(v, str) and v.strip():
            return v, False
    string_vals = [v for v in args.values() if isinstance(v, str) and len(v.strip()) > 20]
    if string_vals: return max(string_vals, key=len), False
    return "", False


def _extract_citation_ids(args: dict, full_text: str) -> Tuple[Set[str], bool]:
    """Returns (ids, used_canonical_field)."""
    if isinstance(args, dict):
        canonical = args.get("cited_artifacts")
        if isinstance(canonical, list):
            ids = {str(c) for c in canonical if isinstance(c, (str, int))}
            ids = {i for i in ids if _ID_PATTERN.fullmatch(i)}
            if ids: return ids, True
        if isinstance(canonical, str) and _ID_PATTERN.fullmatch(canonical):
            return {canonical}, True
        for v in args.values():
            if isinstance(v, list):
                ids = {str(x) for x in v if isinstance(x, str) and _ID_PATTERN.fullmatch(x)}
                if ids: return ids, False
            elif isinstance(v, str) and _ID_PATTERN.fullmatch(v):
                return {v}, False
    return set(_ID_PATTERN.findall(full_text or "")), False


print("✓ Reward helpers ready")

In [ ]:
# ── Reward functions (6 total) ─────────────────────────────────────────────────

_ROUGE = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


# ── R1: Graduated format (5-tier, 0.0 → 1.0) ─────────────────────────────────
def r_format_graduated(completions, **kwargs):
    out = []
    for c in completions:
        text      = _completion_text(c)
        score     = 0.0
        canonical = "```json" in text
        if canonical:              score = 0.2
        elif "<tool_call>" in text: score = 0.1
        parsed = parse_tool_call(text)
        if parsed is None: out.append(score); continue
        score   = 0.4 if canonical else 0.3
        py_call     = bool(re.search(r'\b(?:' + '|'.join(KNOWN_TOOLS) + r')\s*\(', text))
        double_wrap = bool(re.search(r'submit_resolution\s*\([^)]*?submit_resolution\s*\(', text, re.DOTALL))
        if parsed["tool_name"] in KNOWN_TOOLS:
            score = 0.6 if canonical else 0.5
        if parsed["tool_name"] == "submit_resolution":
            args = parsed.get("arguments", {})
            has_resolution = isinstance(args.get("resolution"), str) and len(args["resolution"].strip()) > 20
            if has_resolution:
                score = 0.8 if canonical else 0.65
            req = {"resolution", "cited_artifacts", "confidence"}
            fields_ok = (has_resolution and req.issubset(args.keys())
                         and isinstance(args.get("cited_artifacts"), list))
            if fields_ok and canonical and not py_call: score = 1.0
            elif fields_ok: score = 0.85
        if double_wrap:   score = min(score, 0.35)
        elif py_call:     score = min(score, 0.5)
        out.append(score)
    return out


# ── R2: Resolution quality (ROUGE-L vs gold) ──────────────────────────────────
def r_resolution_quality(completions, ticket_id, **kwargs):
    idx = get_ticket_index()
    out = []
    for c, tid in zip(completions, ticket_id):
        text   = _completion_text(c)
        parsed = parse_tool_call(text)
        if parsed is None or parsed["tool_name"] != "submit_resolution":
            out.append(0.0); continue
        args = parsed.get("arguments", {})
        submitted, used_canonical = _extract_resolution_text(args)
        gold = idx.get(tid, {}).get("gold_resolution", "")
        gold = gold if isinstance(gold, str) else str(gold)
        if not submitted or not gold: out.append(0.0); continue
        f1 = _ROUGE.score(gold, submitted)["rougeL"].fmeasure
        if not used_canonical: f1 *= 0.6
        out.append(f1)
    return out


# ── R3: Citation grounding (F1 + behavioral floor) ────────────────────────────
def r_citation_grounding(completions, ticket_id, prompts=None, **kwargs):
    idx      = get_ticket_index()
    _prompts = list(prompts) if prompts is not None else [None] * len(completions)
    out = []
    for c, tid, pr in zip(completions, ticket_id, _prompts):
        text   = _completion_text(c)
        parsed = parse_tool_call(text)
        ticket = idx.get(tid, {})
        gold   = set(ticket.get("gold_cited_ids", []))
        is_unans = ticket.get("is_unanswerable", False)
        if parsed is None or parsed["tool_name"] != "submit_resolution":
            out.append(0.0); continue
        args  = parsed.get("arguments", {})
        cited, used_canonical = _extract_citation_ids(args, text)
        if is_unans:
            esc = bool(args.get("escalate", False))
            out.append(1.0 if (not cited and esc) else 0.3 if not cited else 0.0); continue
        if not gold:
            esc = bool(args.get("escalate", False))
            out.append(1.0 if esc else 0.1); continue
        if not cited: out.append(0.0); continue
        ctx_ids = set(re.findall(r'### (\S+)', _prompt_user_text(pr))) if pr is not None else set()
        tp = len(cited & gold)
        if tp == 0:
            if ctx_ids:
                out.append(0.3 if bool(cited & ctx_ids) else 0.05)
            else:
                out.append(0.0)
            continue
        p_val = tp / len(cited)
        r_val = tp / len(gold)
        f1    = 2 * p_val * r_val / (p_val + r_val)
        if not used_canonical: f1 *= 0.6
        out.append(0.3 + 0.7 * f1 if (ctx_ids and cited & ctx_ids) else f1)
    return out


# ── R4: Confidence calibration (Brier-style) ──────────────────────────────────
def r_calibration(completions, ticket_id, **kwargs):
    idx = get_ticket_index()
    out = []
    for c, tid in zip(completions, ticket_id):
        text   = _completion_text(c)
        parsed = parse_tool_call(text)
        if parsed is None or parsed["tool_name"] != "submit_resolution":
            out.append(0.0); continue
        args = parsed.get("arguments", {})
        try:
            conf = float(args.get("confidence", 0.5))
            conf = max(0.0, min(1.0, conf))
        except Exception:
            out.append(0.0); continue
        gold       = idx.get(tid, {}).get("gold_resolution", "")
        gold       = gold if isinstance(gold, str) else str(gold)
        submitted, used_canonical = _extract_resolution_text(args)
        quality    = _ROUGE.score(gold, submitted)["rougeL"].fmeasure if (gold and submitted) else 0.0
        if not used_canonical: quality *= 0.6
        out.append(1.0 - (conf - quality) ** 2)
    return out


# ── R5: Parsimony (Goldilocks length: 100-250 words = 1.0) ────────────────────
_PENDING_COMPLETIONS: List[dict] = []

def r_parsimony(completions, **kwargs):
    tids = list(kwargs.get("ticket_id") or [])
    out  = []
    for i, c in enumerate(completions):
        text        = _completion_text(c)
        parsed      = parse_tool_call(text)
        is_finished = text.strip().endswith("```")
        if parsed is None or not is_finished:
            score = 0.0
        else:
            n = len(text.split())
            if n < 60:       score = 0.1
            elif n < 100:    score = 0.5
            elif n <= 250:   score = 1.0
            elif n <= 400:   score = 0.4
            else:            score = 0.0
        out.append(score)
        if i < len(tids):
            _PENDING_COMPLETIONS.append({
                "ticket_id":  tids[i],
                "completion": text[:3000],
                "parsed":     parsed is not None and is_finished,
                "r_parsimony": round(score, 4),
            })
    return out


# ── R6: Repetition penalty (unique cited IDs ratio) ───────────────────────────
def r_repetition_penalty(completions, **kwargs):
    out = []
    for c in completions:
        text = _completion_text(c)
        ids  = _ID_PATTERN.findall(text)
        if not ids: out.append(1.0); continue
        unique_ratio = len(set(ids)) / len(ids)
        out.append(unique_ratio if (len(ids) > 3 and unique_ratio < 0.5) else 1.0)
    return out


REWARD_FUNCS   = [r_format_graduated, r_resolution_quality, r_citation_grounding,
                  r_calibration, r_parsimony, r_repetition_penalty]
REWARD_WEIGHTS = [0.3, 1.5, 1.0, 0.4, 0.4, 0.4]

print("✓ All 6 reward functions defined")
print(f"  weights: {REWARD_WEIGHTS}")

In [ ]:
# ── CompletionDumper callback + smoke-test reward table ───────────────────────

class CompletionDumper(_TrainerCallback):
    """Saves completion batches from _PENDING_COMPLETIONS to JSONL every N steps."""
    def __init__(self, out_dir: Path, dump_every: int = 1):
        self.out_dir    = Path(out_dir)
        self.dump_every = dump_every
        self.out_dir.mkdir(parents=True, exist_ok=True)

    def on_step_end(self, args, state, control, **kwargs):
        if _PENDING_COMPLETIONS and state.global_step % self.dump_every == 0:
            self._flush(state.global_step)

    def on_train_end(self, args, state, control, **kwargs):
        if _PENDING_COMPLETIONS:
            self._flush(state.global_step)

    def _flush(self, step: int):
        global _PENDING_COMPLETIONS
        entries = _PENDING_COMPLETIONS[:]
        _PENDING_COMPLETIONS.clear()
        for e in entries: e.setdefault("step", step)
        path = self.out_dir / f"step_{step:04d}.jsonl"
        with open(path, "w") as fh:
            for e in entries: fh.write(json.dumps(e, ensure_ascii=False) + "\n")
        print(f"  ✓ {len(entries)} completions → {path.name}")


_REWARD_NAMED = [
    ("format",   r_format_graduated),
    ("quality",  r_resolution_quality),
    ("citation", r_citation_grounding),
    ("calib",    r_calibration),
    ("parsim",   r_parsimony),
]
_REWARD_WEIGHT_SUM = sum(REWARD_WEIGHTS[:5])


def print_reward_table(completions_by_ticket: List[tuple]) -> None:
    """Print compact reward breakdown. Accepts (ticket_id, completion[, prompt]) tuples."""
    if not completions_by_ticket:
        print("  (no completions to score)"); return
    tids         = [t[0] for t in completions_by_ticket]
    completions  = [t[1] for t in completions_by_ticket]
    prompts_list = [t[2] if len(t) > 2 else None for t in completions_by_ticket]

    raw: Dict[str, list] = {}
    for name, fn in _REWARD_NAMED:
        try:
            if name == "citation":   raw[name] = fn(completions, ticket_id=tids, prompts=prompts_list)
            elif name in ("quality", "calib"): raw[name] = fn(completions, ticket_id=tids)
            else:                    raw[name] = fn(completions)
        except Exception as e:
            raw[name] = [0.0] * len(completions)
            print(f"  [warn] {name}: {e}")

    names  = [n for n, _ in _REWARD_NAMED]
    totals = [
        sum(raw[names[j]][i] * REWARD_WEIGHTS[j] for j in range(len(names))) / _REWARD_WEIGHT_SUM
        for i in range(len(completions))
    ]

    COL_W, TID_W = 9, 14
    hdr = f"  {'#':<3}  {'ticket_id':<{TID_W}}" + "".join(f"  {n:>{COL_W}}" for n in names) + f"  {'TOTAL':>{COL_W}}"
    sep = "  " + "-" * (len(hdr) - 2)
    print()
    print("  ┌─ Smoke-test reward summary " + "─" * max(0, len(sep) - 30) + "┐")
    print(hdr); print(sep)
    col_sums = {n: 0.0 for n in names}
    for i, (tid, total) in enumerate(zip(tids, totals)):
        row = f"  {i+1:<3}  {tid:<{TID_W}}"
        for n in names:
            v = raw[n][i]; col_sums[n] += v
            row += f"  {v:>{COL_W}.3f}"
        row += f"  {total:>{COL_W}.3f}"
        print(row)
    print(sep)
    mean_total = sum(totals) / len(totals)
    mean_row   = f"  {'avg':<3}  {'':< {TID_W}}" + "".join(f"  {col_sums[n]/len(completions):>{COL_W}.3f}" for n in names) + f"  {mean_total:>{COL_W}.3f}"
    print(mean_row)
    print("  └" + "─" * (len(sep) - 3) + "┘\n")


print("✓ CompletionDumper + reward table ready")

In [ ]:
# ── Plot helpers ──────────────────────────────────────────────────────────────

def save_reward_curve(log_history: list, output_path: Path):
    reward_keys = [
        ("reward",                              "Total reward",       "#2d6a4f", 1.8),
        ("rewards/r_format_graduated/mean",      "Format",             "#9d4edd", 1.0),
        ("rewards/r_resolution_quality/mean",    "Resolution Quality", "#1d3557", 1.2),
        ("rewards/r_citation_grounding/mean",    "Citation Grounding", "#457b9d", 1.2),
        ("rewards/r_calibration/mean",           "Calibration",        "#e76f51", 1.2),
        ("rewards/r_parsimony/mean",             "Parsimony",          "#e9c46a", 1.2),
        ("rewards/r_repetition_penalty/mean",    "Repetition Penalty", "#a8dadc", 1.0),
    ]
    curves = {}
    for entry in log_history:
        step = entry.get("step")
        if step is None: continue
        for key, *_ in reward_keys:
            if key in entry:
                curves.setdefault(key, {"steps": [], "vals": []})
                curves[key]["steps"].append(step)
                curves[key]["vals"].append(entry[key])

    if not curves:
        print(f"  No reward data — skipping plot.")
        print(f"  Keys in last log entry: {list(log_history[-1].keys()) if log_history else 'empty'}")
        return

    fig, ax = plt.subplots(figsize=(11, 5))
    for key, label, colour, lw in reward_keys:
        if key in curves:
            ax.plot(curves[key]["steps"], curves[key]["vals"],
                    label=label, color=colour, linewidth=lw, alpha=0.9,
                    marker="o", markersize=5)
    ax.set_xlabel("Training step", fontsize=12)
    ax.set_ylabel("Mean reward", fontsize=12)
    ax.set_title("TriageAgent — GRPO Smoke Test Reward Curves (5 steps, Qwen2.5-0.5B)",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=9, loc="best")
    ax.grid(axis="y", alpha=0.3)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  ✓ Saved → {output_path}")


print("✓ Plot helpers ready")

In [ ]:
# ── Build dataset ─────────────────────────────────────────────────────────────
print(f"Building dataset from {len(train_tickets)} tickets…")
dataset = build_dataset(train_tickets, get_corpus())
print(f"✓ Dataset ready — {len(dataset)} rows")

# ── GRPOConfig ────────────────────────────────────────────────────────────────
from trl import GRPOConfig, GRPOTrainer
import inspect

warmup_steps = max(1, int(MAX_STEPS * 0.1))

extra_kwargs = {}
try:
    sig = inspect.signature(GRPOConfig)
    if "log_completions" in sig.parameters:
        extra_kwargs["log_completions"]         = True
        extra_kwargs["num_completions_to_print"] = 0
except Exception:
    pass

training_args = GRPOConfig(
    output_dir              = str(OUTPUT_DIR),
    # GRPO group size — 4 generations (half of A100 production value)
    num_generations         = 4,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    # Completion length — 256 tokens fits T4 comfortably
    max_completion_length   = 256,
    # Optimiser
    learning_rate           = 5e-6,
    warmup_steps            = warmup_steps,
    lr_scheduler_type       = "cosine",
    optim                   = "adamw_torch",
    # GRPO variant — dr_grpo fixes the length bias in vanilla GRPO
    loss_type               = "dr_grpo",
    reward_weights          = REWARD_WEIGHTS,
    # Schedule
    max_steps               = MAX_STEPS,
    save_steps              = MAX_STEPS,     # save once at the end
    logging_steps           = 1,
    # Hardware
    bf16                    = True,
    use_vllm                = False,          # no vLLM on T4
    # Generation
    temperature             = 0.9,
    top_p                   = 0.95,
    repetition_penalty      = 1.0,
    # Logging / hub — disabled for smoke test
    report_to               = "none",
    push_to_hub             = False,
    remove_unused_columns   = False,
    **extra_kwargs,
)

# ── Trainer ───────────────────────────────────────────────────────────────────
trainer = GRPOTrainer(
    model        = MODEL,
    reward_funcs = REWARD_FUNCS,
    train_dataset= dataset,
    args         = training_args,
)

trainer.add_callback(CompletionDumper(
    out_dir    = OUTPUT_DIR / "completions",
    dump_every = 1,
))

# ── Train ─────────────────────────────────────────────────────────────────────
print(f"\nStarting GRPO training — model: {MODEL}, steps: {MAX_STEPS}")
print("=" * 60)

try:
    trainer.train()
    trainer.save_model()
    print(f"\n✓ Training complete. Model saved → {OUTPUT_DIR}")
except Exception as e:
    import traceback
    print(f"\n✗ Training failed: {type(e).__name__}: {e}")
    traceback.print_exc()
    try:
        trainer.save_model()
        print(f"✓ Partial model saved → {OUTPUT_DIR}")
    except Exception:
        pass
    raise

In [ ]:
# ── Plot smoke-test reward curves ─────────────────────────────────────────────
log_history = trainer.state.log_history
print(f"log_history entries: {len(log_history)}")
if log_history:
    print(f"Keys in last entry : {list(log_history[-1].keys())}")

save_reward_curve(log_history, PLOTS_DIR / "smoke_reward_curve.png")

In [ ]:
# ── Post-training reward table on 4 tickets ───────────────────────────────────
# Load the trained model (or latest checkpoint) and score a few tickets
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
import torch

print("Loading trained model for reward table…")
eval_tok = AutoTokenizer.from_pretrained(str(OUTPUT_DIR), trust_remote_code=True)
eval_mdl = AutoModelForCausalLM.from_pretrained(
    str(OUTPUT_DIR),
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
eval_mdl.eval()

gen_cfg = GenerationConfig(
    max_new_tokens=256,
    do_sample=False,
    pad_token_id=eval_tok.eos_token_id,
)

corpus      = get_corpus()
eval_pairs  = []
n_eval      = min(4, len(train_tickets))

for t in train_tickets[:n_eval]:
    tid      = t.get("ticket_id", t.get("id", ""))
    messages = build_training_prompt(t, corpus)
    input_ids = eval_tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(eval_mdl.device)
    with torch.no_grad():
        out = eval_mdl.generate(input_ids, generation_config=gen_cfg)
    text = eval_tok.decode(out[0][input_ids.shape[-1]:], skip_special_tokens=True)
    eval_pairs.append((tid, text, messages))
    print(f"  scored {tid}")

print_reward_table(eval_pairs)

In [ ]:
# ── Full A100 training reward curve (200 steps, Qwen2.5-3B) ──────────────────
from IPython.display import Image, display

candidates = [
    PLOTS_DIR / "training_reward_curve.png",
    PLOTS_DIR / "reward_curve.png",
]
shown = False
for p in candidates:
    if p.exists():
        print(f"Full 200-step A100 training run ({p.name}):")
        display(Image(str(p)))
        shown = True
        break
if not shown:
    print("Full curve not found. Available:", [f.name for f in PLOTS_DIR.glob("*.png")])

## What each reward function measures

| Reward | Weight | Description |
|--------|--------|-------------|
| `r_format_graduated` | 0.3 | 5-tier partial credit: markup present → JSON parseable → valid tool name → `submit_resolution` with non-empty `resolution` → all required fields + canonical fence |
| `r_resolution_quality` | 1.5 | ROUGE-L F1 between submitted resolution and gold answer. Fallback keys discounted 0.6× so canonical `resolution` field always wins. |
| `r_citation_grounding` | 1.0 | Citation F1 vs gold-cited KB/INC IDs. Behavioral floor: citing *any* ID from retrieved context = 0.3 (bootstraps gradient). Hallucinated IDs = 0.05. |
| `r_calibration` | 0.4 | `1 − (confidence − ROUGE-L)²` — rewards confidence only when the resolution is actually correct. |
| `r_parsimony` | 0.4 | Goldilocks length: 100–250 words = 1.0. Too short = 0.1–0.5. Rambling (>400 words) = 0.0. |
| `r_repetition_penalty` | 0.4 | Unique-ID ratio — penalises repeating the same KB/INC ID multiple times. |

## Training progress (200-step A100 run, Qwen2.5-3B-Instruct)

| Metric | Steps 1–10 | Steps 150–200 | Δ |
|--------|-----------|---------------|---|
| Calibration | 0.528 | **0.977** | **+0.449** ↑ |
| Parsimony | 0.246 | **0.940** | **+0.694** ↑ |
| Citation Grounding | 0.756 | **0.862** | **+0.106** ↑ |
| Resolution Quality | 0.162 | 0.181 | +0.019 |
| Format Adherence | 1.000 | 0.992 | — |
| Repetition Penalty | 1.000 | 0.990 | — |

The base model already knew the schema. GRPO taught it to be confident only when correct (+45% calibration), to answer concisely (+69% parsimony), and to ground citations in retrieved evidence (+11% grounding).

**Full trained model:** [yahid/triage-agent-qwen3b](https://huggingface.co/yahid/triage-agent-qwen3b)